må ha rank_bm25 
https://www.geeksforgeeks.org/nlp/what-is-bm25-best-matching-25-algorithm/

In [192]:
#pip install rank_bm25

In [211]:
# used claude for debuggig and said this might be the issue with pyarrow and pandas. I don't know if this is the case but it seems to be a common issue. I will try to clear the pyarrow extension types and see if that fixes the issue.
import pyarrow as pa

for ext_type in ["pandas.period", "pandas.interval", "pandas.json"]:
    try:
        pa.unregister_extension_type(ext_type)
    except:
        pass

print("PyArrow extension types cleared")

PyArrow extension types cleared


In [212]:
# importing packages 
import re
import string
import os
from collections import Counter
from typing import List, Dict, Tuple
 
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from rank_bm25 import BM25Okapi
from transformers import pipeline


In [213]:
#  path to the data
PASSAGE_CORPUS_PATH = "data/passage_corpus.csv"
GOLD_MAPPING_PATH = "data/gold_mapping.parquet" # parquet 
OUTPUT_PATH = "results/baseline_results.csv"

# evaluation recall of the k retrieved passages
TOP_K_VALUES = [1, 3, 5, 10, 20]

# model for question answering
QA_MODEL = "deepset/bert-base-cased-squad2"

# number of questions to evaluate on (set to None to evaluate on all questions) but we can set it to a smaller number for testing
MAX_QUESTIONS = None


os.makedirs("results", exist_ok=True)

print("Configuration set!")
print(f"  Passage corpus: {PASSAGE_CORPUS_PATH}")
print(f"  Gold mapping: {GOLD_MAPPING_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Top-k values: {TOP_K_VALUES}")
print(f"  Max questions: {MAX_QUESTIONS if MAX_QUESTIONS else 'All'}")

Configuration set!
  Passage corpus: data/passage_corpus.csv
  Gold mapping: data/gold_mapping.parquet
  Output: results/baseline_results.csv
  Top-k values: [1, 3, 5, 10, 20]
  Max questions: All


In [214]:
# claude said this might fix the issue with conversion
# only hve to run it once to convert the parquet file to csv, then we can use the csv file for the rest of the code. I will keep the parquet file in case we need to go back to it, but I will use the csv file for the rest of the code since it is easier to work with and should not have the same issues with pyarrow and pandas.
PARQUET_PATH = "data/gold_mapping.parquet"
CSV_PATH = "data/gold_mapping.csv"

if os.path.exists(CSV_PATH):
    print(f"✓ {CSV_PATH} already exists, skipping conversion")
else:
    print(f"Converting {PARQUET_PATH} to CSV...")
    
    # Load parquet
    gold_df = pd.read_parquet(PARQUET_PATH)
    
    # Convert list/array columns to strings for CSV storage
    if 'gold_passage_ids' in gold_df.columns:
        gold_df['gold_passage_ids'] = gold_df['gold_passage_ids'].apply(
            lambda x: str(list(x)) if isinstance(x, np.ndarray) else str(x)
        )
    
    if 'gold_urls' in gold_df.columns:
        gold_df['gold_urls'] = gold_df['gold_urls'].apply(
            lambda x: str(list(x)) if isinstance(x, (list, np.ndarray)) else str(x)
        )
    
    # Save as CSV
    gold_df.to_csv(CSV_PATH, index=False)
    print(f"✓ Saved {CSV_PATH} ({len(gold_df)} rows)")
    print(f"  Columns: {list(gold_df.columns)}")

Converting data/gold_mapping.parquet to CSV...


ArrowKeyError: No type extension with name arrow.py_extension_type found

# preprocessing text

https://www.geeksforgeeks.org/python/re-findall-in-python/

In [195]:
def tokenize(text: str) -> List[str]:
    # simple tokenization for BM25 

    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and split
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

In [196]:
def normalize_answer(text: str) -> str:
    # normalize the answer for evaluation and this aslo follow with the tokenization for BM25

    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text) # removing a, an, the 

    def remove_punctuation(text):
        return ''.join(ch for ch in text if ch not in string.punctuation) # removing punctuation

    def remove_whitespace(text):
        return ' '.join(text.split())  # removing extra whitespace ( if there are multiple spaces, it will be reduced to a single space)
    
    return remove_whitespace(remove_punctuation(remove_articles(text.lower()))) 


# Evaluation metric 


In [197]:
def compute_exact_match(prediction: str, ground_truth: str) -> float:
    # compute exact match score between the prediction and the ground truth answer

    return float(normalize_answer(prediction) == normalize_answer(ground_truth))

In [198]:
def compute_f1(prediction: str, ground_truth: str) -> float:
    
    # compute F1 score between the prediction and the ground truth answer
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()
    
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    
    common = Counter(pred_tokens) & Counter(gold_tokens) # count the number of common tokens between the prediction and the ground truth answer
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1


In [199]:
def compute_recall_at_k(retrieved_passages: List[int], gold_ids: List[int], k: int) -> float:

    top_k_retrieved = set(retrieved_passages[:k]) # get the top k retrieved passages
    gold_ids_set = set(gold_ids) # convert gold ids to a set

    return float(len(top_k_retrieved & gold_ids_set) > 0) # compute the recall at k by checking if there is any intersection between the top k retrieved passages and the gold passage ids, and return 1.0 if there is at least one match, otherwise return 0.0
    


    """if len(gold_ids_set) == 0:
        return 0.0

    recall = len(top_k_retrieved & gold_ids_set) / len(gold_ids_set)
    return recall # compute the recall at k by calculating the intersection of the top k retrieved passages and the gold passage ids, and dividing by the number of gold passage ids

    # could use this and just remove the return statement above if we want to compute the recall as the proportion of gold passage ids that are retrieved in the top k, but for this task we only care about whether at least one gold passage id is retrieved in the top k, so we can return 1.0 if there is at least one match and 0.0 otherwise
"""


## the retriver BM25

Very relevant use this 
https://stackoverflow.com/questions/61877065/implementation-of-okapi-bm25-in-python
https://github.com/crawlab-team/bm25/blob/main/bm25okapi.go
https://docs.langchain.com/oss/python/integrations/retrievers/bm25 # okapi for wikipedia 

Somewhat relevant: 
https://github.com/Rohith-2/bm25-fusion/blob/main/src/bm25_fusion/core.py

Not so relevant but take a look at this for furher reseach
https://github.com/xhluca/bm25s/tree/main/bm25s
https://developers.llamaindex.ai/python/framework/integrations/retrievers/bm25_retriever/   # from llama_index.retrievers.bm25 import BM25Retriever



"Okapi BM25" is the standard, variations exist to improve it:BM25F: A version that handles multiple fields (e.g., title, body, anchor text) with different importance.BM25+: Developed to fix a deficiency where long documents that match the query term are unfairly scored compared to shorter documents.rank_bm25 (Python): A popular Python library that includes implementations like BM25Okapi, BM25L, and BM25Plus

In [200]:
class BM25retriever:

    #BM25 retriever over a passage corpus 
    def __init__(self, passage_df: pd.DataFrame):

        self.passage_df = passage_df.reset_index(drop=True) 
        self.passage_ids = passage_df['passage_id'].tolist()


        self.tokenized_corpus = [tokenize(text) for text in tqdm(passage_df['text'].tolist(), desc="Tokenizing")]

        
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        print(f"  Index built with {len(self.passage_ids)} passages") # initialize the BM25 retriever with the tokenized passage corpus


    def retrieve(self, query: str, top_k: int=10) -> List[Tuple[int,float,str]]:
        tokenized_query = tokenize(query) # tokenize the query
        scores = self.bm25.get_scores(tokenized_query) # get BM25 scores for the query against all passages

        top_k_indices = np.argsort(scores)[::-1][:top_k] # get the indices of the top k passages based on the BM25 scores

        # compile the results for the top k retrieved passages, including the passage id, BM25 score, and passage text
        results = []
        for idx in top_k_indices:
            passage_id = self.passage_ids[idx] # get the passage id for the retrieved passage
            score = scores[idx] # get the BM25 score for the retrieved passage
            text = self.passage_df.loc[idx, 'text'] # get the text of the retrieved passage
            results.append((passage_id, score, text)) # append the passage id, score, and text of the retrieved passage to the results list
        return results



# BERT QA (baseline)

In [201]:
class BertQA:
    def __init__(self, model_name: str = QA_MODEL):
        self.qa_pipeline = pipeline("question-answering", model=model_name, device=0) # initialize the question answering pipeline with the specified model

    def answer_question(self, question: str, context: str) -> Dict:
        try:
            result = self.qa_pipeline(question=question, context=context)
            return {
                'answer': result['answer'],
                'score': result['score'],
                'start': result['start'],
                'end': result['end']
            }
        except Exception as e:
            # Handle edge cases (empty context, etc.)
            return {
                'answer': '',
                'score': 0.0,
                'start': 0,
                'end': 0
            }

# main eval 

In [202]:
def run_baseline_evaluation(
        passage_df: pd.DataFrame,
        gold_df: pd.DataFrame,
        top_k_values: List[int] = TOP_K_VALUES,
        max_questions: int = None
) -> pd.DataFrame:
    
    if max_questions: 
        gold_df = gold_df.head(max_questions) # limit the number of questions to evaluate on if max_questions is specified
    print(f"\n{'='*60}")
    print("FRAMES Baseline Evaluation")
    print(f"{'='*60}")
    print(f"Passages: {len(passage_df)}")
    print(f"Questions: {len(gold_df)}")
    print(f"Top-k values: {top_k_values}")
    print(f"{'='*60}\n")
    
    retriever = BM25retriever(passage_df) # initialize the BM25 retriever with the passage dataframe
    qa_model = BertQA() # initialize the BERT question answering model

    # storing the results 
    results = []
    max_k = max(top_k_values) # get the maximum k value for retrieval   


    print("Evaluating questions...")
    for idx, row in tqdm(gold_df.iterrows(), total=len(gold_df), desc="Evaluating"):
        question = row["promt"]
        gold_answer = row["answer"]
        gold_passage_ids = row["passage_ids"] # get the gold passage ids for

        # retriving passage 
        retrived = retriever.retrieve(question, top_k=max_k) # retrieve the top k passages for the question using the BM25 retriever
        retrieved_ids = [pid for pid, _, _ in retrived] # extract

        # compute recall for each k 
        recall_score = {}
        for k in top_k_values:
            recall_score[f"recall@{k}"] = compute_recall_at_k(
                retrieved_ids, gold_passage_ids, k) # compute the recall at k for the retrieved passages and the gold passage ids, and store the result in the recall_score dictionary with the key 'recall@k'
            
        # extracting the asnwers for top 1 pasage
        if retrived:
            top_passage_id, top_score, top_passage_text = retrived[0] # get the passage id, BM25 score, and text of the top retrieved passage
            qa_result = qa_model.answer_question(question, top_passage_text) # extract the answer for the question using the BERT question answering model with the top retrieved passage as context
            predicted_answer = qa_result['answer'] # get the predicted answer from the QA result
            qa_confidence = qa_result['score'] # get the QA confidence score from the QA result
        else:
            predicted_answer = '' # if no passages were retrieved, set the predicted answer to an empty string
            qa_confidence = 0.0 # if no passages were retrieved, set the QA confidence score to 0.0
            top_passage_id = None # if no passages were retrieved, set the top passage id to None
            top_passage_text = None # if no passages were retrieved, set the top passage text

        # compute evaluation metrics and f1 
        em_score = compute_exact_match(predicted_answer, gold_answer) # compute the exact match score between the predicted answer and the gold answer
        f1 = compute_f1(predicted_answer, gold_answer) # compute the F1 score between the predicted answer and the gold answer

        # storing the results 
        results = {
            "question_id": row["question_id"],
            "question": question,
            "gold_answer": gold_answer,
            "predicted_answer": predicted_answer,
            "em": em_score,
            "f1": f1,
            "qa_confidence": qa_confidence,
            "top_passage_id": top_passage_id,
            "n_gold_passages": len(gold_passage_ids),
            **recall_score # unpack the recall scores for each k and add them to the results

        }
        results.append(results) # append the results for the current question to the overall results list

    results_df = pd.DataFrame(results) # convert the results list to a pandas DataFrame
    return results_df

In [203]:
def print_summary(results_df: pd.DataFrame, top_k_values: List[int]):
    # print evaluation summary statistics
    print(f"\n{'='*60}")
    print("EVALUATION RESULTS")
    print(f"{'='*60}")
    
    # print retrieval performance for each top-k value
    print("\n--- Retrieval Performance (BM25) ---")
    for k in top_k_values:
        col = f'recall@{k}'
        mean_recall = results_df[col].mean()
        print(f"  Recall@{k}: {mean_recall:.4f} ({mean_recall*100:.1f}%)")
    
    print("\n--- QA Performance (BERT) ---")
    print(f"  Exact Match (EM): {results_df['em'].mean():.4f} ({results_df['em'].mean()*100:.1f}%)")
    print(f"  F1 Score:         {results_df['f1'].mean():.4f} ({results_df['f1'].mean()*100:.1f}%)")
    
    print("\n--- Error Analysis ---")
    # Retrieval failures: gold passage not in top-k
    retrieval_failures = (results_df['recall@10'] == 0).sum()
    print(f"  Retrieval failures (gold not in top-10): {retrieval_failures} ({retrieval_failures/len(results_df)*100:.1f}%)")
    
    # Cases where retrieval succeeded but QA failed
    retrieval_success_qa_fail = ((results_df['recall@10'] == 1) & (results_df['em'] == 0)).sum()
    print(f"  Retrieval OK but QA failed: {retrieval_success_qa_fail} ({retrieval_success_qa_fail/len(results_df)*100:.1f}%)")
    
    print(f"\n{'='*60}\n")

In [204]:
def analyze_by_reasoning_type(results_df: pd.DataFrame, gold_df: pd.DataFrame):
    """
    Break down performance by reasoning type.
    """
    # Merge reasoning types into results
    # This requires the original gold_df to have reasoning types
    # You may need to join with the preprocessed FRAMES data
    
    print("\n--- Performance by Reasoning Type ---")
    print("(To implement: merge with reasoning_types column from preprocessed data)")
    


Main entry point 

In [205]:
def main():
    """
    Main function to run the baseline evaluation.
    """
    import os
    import ast
    
    # Create results directory
    os.makedirs("results", exist_ok=True)
    
    # Load data
    print("Loading data...")
    
    # Load passages from CSV
    passages_df = pd.read_csv(PASSAGE_CORPUS_PATH)
    
    # Load gold mapping from parquet
    gold_df = pd.read_parquet(GOLD_MAPPING_PATH)
    
    # Handle gold_passage_ids - might be stored as numpy array
    if 'gold_passage_ids' in gold_df.columns:
        first_val = gold_df['gold_passage_ids'].iloc[0]
        if isinstance(first_val, np.ndarray):
            gold_df['gold_passage_ids'] = gold_df['gold_passage_ids'].apply(list)
    
    print(f"  Loaded {len(passages_df)} passages")
    print(f"  Loaded {len(gold_df)} questions")
    
    # Run evaluation
    results_df = run_baseline_evaluation(
        passage_df=passages_df,
        gold_df=gold_df,
        top_k_values=TOP_K_VALUES,
        max_questions=MAX_QUESTIONS
    )
    
    # Print summary
    print_summary(results_df, TOP_K_VALUES)
    
    # Save results
    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Results saved to {OUTPUT_PATH}")
    
    return results_df


if __name__ == "__main__":
    results = main()

Loading data...


ArrowKeyError: No type extension with name arrow.py_extension_type found